# A2 RNN's Visual encoder and Calculator.

# Function definitions for creating the datasets

First we need to create our datasets that are going to be used for training our models.

In order to create image queries of simple arithmetic operations such as '15+13' or '42-10' we need to create images of '+' and '-' signs using ***open-cv*** library. We will use these operand signs together with the MNIST dataset to represent the digits.

## Given functions

In [1]:
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2
import numpy as np
import tensorflow as tf
import random
from sklearn.model_selection import train_test_split


from tensorflow.keras.layers import Dense, RNN, LSTM, Flatten, TimeDistributed, LSTMCell
from tensorflow.keras.layers import RepeatVector, Conv2D, SimpleRNN, GRU, Reshape, ConvLSTM2D, Conv2DTranspose

2026-01-06 21:58:48.819220: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-06 21:58:48.852273: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-06 21:59:07.830747: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
from scipy.ndimage import rotate
tf.random.set_seed(42)


# Create plus/minus operand signs
def generate_images(number_of_images=50, sign='-'):
    blank_images = np.zeros([number_of_images, 28, 28])  # Dimensionality matches the size of MNIST images (28x28)
    x = np.random.randint(12, 16, (number_of_images, 2)) # Randomized x coordinates
    y1 = np.random.randint(6, 10, number_of_images)       # Randomized y coordinates
    y2 = np.random.randint(18, 22, number_of_images)     # -||-

    for i in range(number_of_images): # Generate n different images
        cv2.line(blank_images[i], (y1[i], x[i,0]), (y2[i], x[i, 1]), (255,0,0), 2, cv2.LINE_AA)     # Draw lines with randomized coordinates
        if sign == '+':
            cv2.line(blank_images[i], (x[i,0], y1[i]), (x[i, 1], y2[i]), (255,0,0), 2, cv2.LINE_AA) # Draw lines with randomized coordinates

    return blank_images

def show_generated(images, n=5):
    plt.figure(figsize=(2, 2))
    for i in range(n**2):
        plt.subplot(n, n, i+1)
        plt.axis('off')
        plt.imshow(images[i])
    plt.show()

In [3]:
def create_data(highest_integer, num_addends=2, operands=['+', '-']):
    """
    Creates the following data for all pairs of integers up to [1:highest integer][+/-][1:highest_integer]:

    @return:
    X_text: '51+21' -> text query of an arithmetic operation (5)
    X_img : Stack of MNIST images corresponding to the query (5 x 28 x 28) -> sequence of 5 images of size 28x28
    y_text: '72' -> answer of the arithmetic text query
    y_img :  Stack of MNIST images corresponding to the answer (3 x 28 x 28)

    Images for digits are picked randomly from the whole MNIST dataset.
    """

    num_indices = [np.where(MNIST_labels==x) for x in range(10)]
    num_data = [MNIST_data[inds] for inds in num_indices]
    image_mapping = dict(zip(unique_characters[:10], num_data))
    image_mapping['-'] = generate_images()
    image_mapping['+'] = generate_images(sign='+')
    image_mapping['*'] = generate_images(sign='*')
    image_mapping[' '] = np.zeros([1, 28, 28])

    X_text, X_img, y_text, y_img = [], [], [], []

    for i in range(highest_integer + 1):      # First addend
        for j in range(highest_integer + 1):  # Second addend
            for sign in operands: # Create all possible combinations of operands
                query_string = to_padded_chars(str(i) + sign + str(j), max_len=max_query_length, pad_right=True)
                query_image = []
                for n, char in enumerate(query_string):
                    image_set = image_mapping[char]
                    index = np.random.randint(0, len(image_set), 1)
                    query_image.append(image_set[index].squeeze())

                result = eval(query_string)
                result_string = to_padded_chars(result, max_len=max_answer_length, pad_right=True)
                result_image = []
                for n, char in enumerate(result_string):
                    image_set = image_mapping[char]
                    index = np.random.randint(0, len(image_set), 1)
                    result_image.append(image_set[index].squeeze())

                X_text.append(query_string)
                X_img.append(np.stack(query_image))
                y_text.append(result_string)
                y_img.append(np.stack(result_image))

    return np.stack(X_text), np.stack(X_img)/255., np.stack(y_text), np.stack(y_img)/255.

def to_padded_chars(integer, max_len=3, pad_right=False):
    """
    Returns a string of len()=max_len, containing the integer padded with ' ' on either right or left side
    """
    length = len(str(integer))
    padding = (max_len - length) * ' '
    if pad_right:
        return str(integer) + padding
    else:
        return padding + str(integer)


## My functions

In [6]:
# Teacher forcing preparation
vocabulary_tf = list(unique_characters)+['<start>','<end>'] 

indices = {char:i for i, char in enumerate(vocabulary_tf)}
reverse_indices={i:char for i,char in enumerate(vocabulary_tf)}


# One-hot encoding
def encode_labels_tf(labels, vocabulary=vocabulary_tf, indices_map=indices):
  n = len(labels)
  length = len(labels[0])+2 # for start of sequence <start> and end of sequence <end> tokens.

  one_hot = np.zeros([n, length, len(vocabulary)])
  for i, label in enumerate(labels):
    full_label = ['<start>'] + list(label) + ['<end>']
    m = np.zeros([length, len(vocabulary)])
    for j, char in enumerate(full_label):
       m[j, indices_map[char]] = 1
    one_hot[i] = m

  return one_hot

# One-hot decoding
def decode_labels_tf(labels, indices_map=reverse_indices):

    pred_indices = np.argmax(labels, axis=-1) 
    
    decoded_list = []
    for sequence in pred_indices:

        chars = [indices_map[i] for i in sequence if indices_map[i] not in ['<start>', '<end>', '<pad>']]
        decoded_list.append(''.join(chars))
        
    return decoded_list

X_text_onehot = encode_labels_tf(X_text)
y_text_onehot = encode_labels_tf(y_text)

print(X_text_onehot.shape, y_text_onehot.shape)

(20000, 7, 15) (20000, 5, 15)


In [7]:
# Masking layer for masked teacher forcing
def random_mask_layer(x, training=None, prob=0.5):

    def mask_logic():

        condition = tf.random.uniform([]) > prob
        return tf.cond(condition, lambda: x, lambda: tf.zeros_like(x))


    return tf.cond(tf.cast(training, tf.bool), 
                   mask_logic, 
                   lambda: tf.zeros_like(x))

from tensorflow.keras.layers import Lambda

# Define a global variable to track sampling probability
# We start at 1.0 (Full Teacher Forcing) and move toward 0.0 (Autoregressive)
sampling_prob = tf.Variable(1.0, trainable=False, dtype=tf.float32)
# Updated function to accept the 'p' parameter dynamically
def scheduled_mask_layer(x, training=None, prob=1.0):
    def mask_logic():
        # Use the passed-in probability 'p'
        condition = tf.random.uniform([]) < prob
        return tf.cond(condition, lambda: x, lambda: tf.zeros_like(x))

    # During evaluation (training=False), always return zeros to 
    # force autoregressive behavior
    return tf.cond(tf.cast(training, tf.bool), 
                   mask_logic, 
                   lambda: tf.zeros_like(x))

I0000 00:00:1767733202.838967 2154167 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6127 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060, pci bus id: 0000:01:00.0, compute capability: 8.9


In [8]:
class ScheduledSamplingCallback(tf.keras.callbacks.Callback):
    def __init__(self, prob_var, decay_rate=0.05, min_prob=0.1):
        super().__init__()
        self.prob_var = prob_var
        self.decay_rate = decay_rate
        self.min_prob = min_prob

    def on_epoch_end(self, epoch, logs=None):
        # Calculate the new value
        current_val = self.prob_var.numpy()
        new_val = max(self.min_prob, current_val - self.decay_rate)
        
        # Update the TensorFlow variable
        tf.keras.backend.set_value(self.prob_var, new_val)
        
        # Log to console for monitoring
        print(f"\n --- End of Epoch {epoch + 1}: Teacher Forcing Ratio set to {new_val:.2f} ---")

In [ ]:
# 1. Ensure the global variable exists
sampling_prob = tf.Variable(1.0, trainable=False, dtype=tf.float32)

# 2. Modify train_warmup to include the callback
def train_warmup(model, x, y, val_data, learning_rate=4.0e-4, weight_decay=1.0e-2):
    optimizer = tf.keras.optimizers.AdamW(learning_rate=learning_rate, weight_decay=weight_decay)
    loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)
    model.compile(loss=loss, optimizer=optimizer, metrics=['categorical_accuracy'])

    # Instantiate the callback here
    # We decay faster in warmup to get the model used to its own errors early
    sampling_cb = ScheduledSamplingCallback(sampling_prob, decay_rate=0.04, min_prob=0.5)
    
    early_stopper = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True
    )

    history = model.fit(
        x=x, y=y,
        epochs=50,
        batch_size=32,
        validation_data=val_data,
        callbacks=[early_stopper, sampling_cb]  # Add it here
    )
    return history

# 3. Modify train_fine_tune to continue the decay
def train_fine_tune(model, x, y, val_data, learning_rate=1.0e-5, weight_decay=5.0e-2):
    optimizer = tf.keras.optimizers.AdamW(learning_rate=learning_rate, weight_decay=weight_decay)
    loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)
    model.compile(loss=loss, optimizer=optimizer, metrics=['categorical_accuracy'])

    # Start from current sampling_prob and decay to near zero
    sampling_cb = ScheduledSamplingCallback(sampling_prob, decay_rate=0.02, min_prob=0.05)

    lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)
    early_stopper = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

    history = model.fit(
        x=x, y=y, 
        epochs=60,
        batch_size=32,
        validation_data=val_data,
        callbacks=[lr_scheduler, early_stopper, sampling_cb], # Add it here
        verbose=1
    )
    return history

"""# Warmup training function for ease of use
def train_warmup(
        model,
        x,
        y,
        val_data,
        learning_rate=4.0e-4, # Initial LR
        weight_decay=1.0e-2  # Decoupled Weight Decay
):
    ## Compile
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=learning_rate,
        weight_decay=weight_decay,
    )

    loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

    model.compile(
        loss=loss, 
        optimizer=optimizer, 
        metrics=['categorical_accuracy'])

    ## Training
    stopper_patience_warmup = 5

    early_stopper = tf.keras.callbacks.EarlyStopping(
        monitor = 'val_loss',
        patience=stopper_patience_warmup,
        restore_best_weights=True
    )

    ### Warmup
    history = model.fit(x=x, y=y,
                epochs = 50,
                batch_size = 32,
                validation_data = val_data,
                callbacks = [early_stopper]) # only use an early stopper

    return history

# Fine-tune training function
def train_fine_tune(
        model,
        x,
        y,
        val_data,
        learning_rate=1.0e-5, # Initial LR
        weight_decay=5.0e-2,  # Decoupled Weight Decay
        scheduler_patience = 5,
        stopper_patience = 20
):
    # Fine tune training
    ## Recompile so AdamW moments are reset
    optimizer = tf.keras.optimizers.AdamW(learning_rate=learning_rate, weight_decay=weight_decay)
    loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

    model.compile(
        loss=loss, 
        optimizer=optimizer, 
        metrics=['categorical_accuracy'])

    early_stopper = tf.keras.callbacks.EarlyStopping(
        monitor = 'val_loss',
        patience=stopper_patience,
        restore_best_weights=True
    )

    lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=scheduler_patience,
        min_lr=1e-6,
        verbose=1
    )

    history = model.fit(x=x, y=y, 
                epochs = 60,
                batch_size = 32,
                validation_data = val_data,
                callbacks=[lr_scheduler, early_stopper],
                verbose=1)
    
    return history

"""

"# Warmup training function for ease of use\ndef train_warmup(\n        model,\n        x,\n        y,\n        val_data,\n        learning_rate=4.0e-4, # Initial LR\n        weight_decay=1.0e-2  # Decoupled Weight Decay\n):\n    ## Compile\n    optimizer = tf.keras.optimizers.AdamW(\n        learning_rate=learning_rate,\n        weight_decay=weight_decay,\n    )\n\n    loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)\n\n    model.compile(\n        loss=loss, \n        optimizer=optimizer, \n        metrics=['categorical_accuracy'])\n\n    ## Training\n    stopper_patience_warmup = 5\n\n    early_stopper = tf.keras.callbacks.EarlyStopping(\n        monitor = 'val_loss',\n        patience=stopper_patience_warmup,\n        restore_best_weights=True\n    )\n\n    ### Warmup\n    history = model.fit(x=x, y=y,\n                epochs = 50,\n                batch_size = 32,\n                validation_data = val_data,\n                callbacks = [early_stopper]) # only us

# Creating our data

In [10]:
# Illustrate the generated query/answer pairs

unique_characters = '0123456789+- '       # All unique characters that are used in the queries (13 in total: digits 0-9, 2 operands [+, -], and a space character ' '.)
highest_integer = 99                      # Highest value of integers contained in the queries

max_int_length = len(str(highest_integer))# Maximum number of characters in an integer
max_query_length = max_int_length * 2 + 1 # Maximum length of the query string (consists of two integers and an operand [e.g. '22+10'])
max_answer_length = 3    # Maximum length of the answer string (the longest resulting query string is ' 1-99'='-98')

# Create the data (might take around a minute)
(MNIST_data, MNIST_labels), _ = tf.keras.datasets.mnist.load_data()
X_text, X_img, y_text, y_img = create_data(highest_integer)
print(X_text.shape, X_img.shape, y_text.shape, y_img.shape)




(20000,) (20000, 5, 28, 28) (20000,) (20000, 3, 28, 28)


In [11]:
# Creating visual encoder training data, including ground-truth sequences used for teacher forcing.
size=0.1

X_train_pt, X_test_pt, y_train_pt, y_test_pt = train_test_split(
    X_img, X_text_onehot, random_state=42, test_size=size
)

X_train_pt, X_val_pt, y_train_pt, y_val_pt = train_test_split(
    X_train_pt, y_train_pt, random_state=42, test_size=size/(1-size)
)

max_answer_length_pt = 6 # max anser length while using teacher forcing. pt = pretraining (old name)

y_train_in_pt = y_train_pt[:, :-1, :]
y_train_target_pt = y_train_pt[:, 1:, :]

y_val_in_pt = y_val_pt[:, :-1, :]
y_val_target_pt = y_val_pt[:, 1:, :]

y_test_in_pt = y_test_pt[:, :-1, :]
y_test_target_pt = y_test_pt[:, 1:, :]

In [12]:
# Calculator model data
size=0.1

X_train_calc, X_test_calc, y_train_calc, y_test_calc = train_test_split(
    X_text_onehot[:, :-1, :], y_text_onehot, random_state=42, test_size=size
)

X_train_calc, X_val_calc, y_train_calc, y_val_calc = train_test_split(
    X_train_calc, y_train_calc, random_state=42, test_size=size/(1-size)
)

max_answer_length_tf = 4 # max anser length while using teacher forcing

y_train_in_calc = y_train_calc[:, :-1, :]
y_train_target_calc = y_train_calc[:, 1:, :]

y_val_in_calc = y_val_calc[:, :-1, :]
y_val_target_calc = y_val_calc[:, 1:, :]

y_test_in_calc = y_test_calc[:, :-1, :]
y_test_target_calc = y_test_calc[:, 1:, :]

# Visual encoder model

### Building visual encoder

In [13]:
# Your code is: damn code

from tensorflow.keras.layers import Lambda, BatchNormalization, Activation, MaxPooling2D, LSTM, TimeDistributed, Dropout, Input, Add, LayerNormalization, Attention, Concatenate,GlobalAveragePooling2D
from tensorflow.keras.regularizers import L2, L1L2
from tensorflow.keras import layers, Sequential


# First we create a build-encoder function
def build_image2text_encoder(dropout, RLstrength, max_size=512): #old name which is an artifact of earlier ideas.
    
    # Initialize an encoder
    X_in = Input(shape = (5,28,28,1)) # 5 times a grayscale image

    data_augmentation = Sequential([
    layers.RandomRotation(0.05), # Rotate by ~18 degrees
    layers.RandomTranslation(height_factor=0.1, width_factor=0.1), # Shift
    layers.RandomZoom(0.1), # Zoom in/out
    ], name="spatial_augmentation")

    augmentation_layer = layers.TimeDistributed(data_augmentation)(X_in)

    # Build encoder layers
    ## Block 1
    B1 = TimeDistributed(Conv2D(32, (3,3), padding = "same", kernel_regularizer=L2(RLstrength/8)))(augmentation_layer)
    B1 = TimeDistributed(BatchNormalization())(B1)
    B1 = TimeDistributed(Activation('relu'))(B1)
    B1 = TimeDistributed(Dropout(dropout))(B1)
    B1_final = TimeDistributed(MaxPooling2D())(B1)


    ## Initialize the residual connection
    #residual = TimeDistributed(Activation('linear', name = "residual_branch"))(B1_final)
    residual2= TimeDistributed(Conv2D(64, (1,1), kernel_regularizer=L2(RLstrength/8), name = "residual_branch"))(B1_final)
    
    ## Block 2
    B2 = TimeDistributed(Conv2D(64, (3,3), padding = "same", kernel_regularizer=L2(RLstrength/8)))(B1_final)
    B2 = TimeDistributed(BatchNormalization())(B2)
    B2 = TimeDistributed(Activation('relu'))(B2)
    B2_final = TimeDistributed(Dropout(dropout))(B2)

    ## Connect residual connection to output of two Conv2D blocks
    combined = Add()([B2_final, residual2])
    combined = TimeDistributed(Activation('relu'))(combined)

    ## Final pooling before the ConvLSTM2D layer
    final_pooling = TimeDistributed(MaxPooling2D(name='final_pooling'))(combined)


    ## Recurrent convolutional layers
    output, hidden, cell = ConvLSTM2D(
            filters=int(max_size/2), 
            kernel_size=(3,3), 
            padding='same',
            return_sequences=True, 
            use_bias=True, 
            return_state=True, 
            name='ConvLSTM', 
            dropout=dropout, #dropout
            #recurrent_dropout=dropout, #dropout
            kernel_regularizer = L2(RLstrength),
            recurrent_regularizer = L2(RLstrength))(final_pooling) #L2(RLstrength)

    encoder = tf.keras.Model(inputs=X_in, outputs=[output, hidden, cell], name = "encoder_model")
    return encoder

In [14]:
def build_image2text_pretraining(dropout = 0.5, max_size=512,RLstrength=1.0e-4):

    vocab_size = 15

    X_in = Input(shape = (5,28,28,1), name = 'sequence')
    Y_in = Input(shape=(6, vocab_size))
    masked_ground_truth = Lambda(
    lambda x, training: scheduled_mask_layer(x, training=training, prob=sampling_prob),
    output_shape=(4, 15), # Note: Use (4, 15) for calculator, (6, 15) for visual encoder
    name="scheduled_masking_layer"
    )(Y_in)

    encoder = build_image2text_encoder(dropout,RLstrength, max_size=max_size)
    _, hidden, cell = encoder(X_in)
    
    h_flattened = GlobalAveragePooling2D(name='h_flattened')(hidden)#Flatten()(hidden)
    h_initial = Dense(max_size, kernel_regularizer=L2(RLstrength), name='h0')(h_flattened)

    c_flattened = GlobalAveragePooling2D(name='c_flattened')(cell)#Flatten(name='c_flattened')(cell)
    c_initial = Dense(max_size, kernel_regularizer=L2(RLstrength),name='c0')(c_flattened)

    ini_state = [h_initial, c_initial]
    
    output, hidden, cell = LSTM(
        max_size, 
        return_sequences = True,
        return_state=True, 
        dropout = dropout, 
        #recurrent_dropout = dropout, 
        name='lstm_pre_training',
        kernel_regularizer = L2(RLstrength/4),
        recurrent_regularizer = L2(RLstrength/4)
        )(masked_ground_truth, initial_state = ini_state)
        
    dense = TimeDistributed(Dense(vocab_size, activation='softmax', name = 'decoder_dense_pre_training'))
    y_out = dense(output)

    full = tf.keras.Model(inputs=[X_in, Y_in], outputs = y_out, name = 'pretraining_model')
#    loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)
#    full.compile(
#        loss=loss, optimizer=Adam(learning_rate=learning_rate), metrics=['categorical_accuracy']
#    )

    full.summary(expand_nested=True)
    return full



### Training visual encoder

In [15]:
# Training the visual encoder. We train this beforehand.

## Building model
dropout=0.5
RLstrength=0
max_size=256 # Must be 128 at minimum. Must be an even number.
image2text_pretraining = build_image2text_pretraining(dropout=dropout, RLstrength=RLstrength, max_size=max_size)
sampling_prob.assign(1.0)
x=[X_train_pt, y_train_in_pt]
y=y_train_target_pt
val_data = ([X_val_pt, y_val_in_pt], y_val_target_pt)

learning_rate=4.0e-4 # Initial LR
weight_decay=1.0e-2  # Decoupled Weight Decay

train_warmup(image2text_pretraining,
             x = x,
             y = y,
             val_data = val_data,
             learning_rate=learning_rate,
             weight_decay=weight_decay)


Model: "pretraining_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence            │ (None, 5, 28, 28, │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_model       │ [(None, 5, 7, 7,  │    906,560 │ sequence[0][0]    │
│ (Functional)        │ 128), (None, 7,   │            │                   │
│                     │ 7, 128), (None,   │            │                   │
│                     │ 7, 7, 128)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └ input_layer_1  │ (None, 5, 28, 28, │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 28, 28, │          0 │ -                 │
│ time_distributed    │ 1)                │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 28, 28, │        320 │ -                 │
│ time_distributed_1  │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 28, 28, │        128 │ -                 │
│ time_distributed_2  │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 28, 28, │          0 │ -                 │
│ time_distributed_3  │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 28, 28, │          0 │ -                 │
│ time_distributed_4  │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │          0 │ -                 │
│ time_distributed_5  │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │     18,496 │ -                 │
│ time_distributed_7  │ 64)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │        256 │ -                 │
│ time_distributed_8  │ 64)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │          0 │ -                 │
│ time_distributed_9  │ 64)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │          0 │ -                 │
│ time_distributed_10 │ 64)               │            │                 

 Total params: 1,254,991 (4.79 MB)

 Trainable params: 1,254,799 (4.79 MB)

 Non-trainable params: 192 (768.00 B)

Epoch 1/50


E0000 00:00:1767733275.791360 2154167 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: MutableGraphView::SortTopologically error: detected edge(s) creating cycle(s) {'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_100/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/mul_4' -> 'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_100/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/add_7', 'StatefulPartitionedCall/gradient_tape/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/pretraining_model_1/encoder_model_1/ConvLSTM_1/while_grad/body/_353/gradient_tape/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/gradients/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/Tanh_1_grad/TanhGrad' -> 'StatefulPartitionedCall/gradient_tape/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/pretraining_model_1/encoder_model_1/ConvLSTM_1/while_grad/body/_353/g

500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - categorical_accuracy: 0.3389 - loss: 2.0224

W0000 00:00:1767733300.384611 2154167 loop_optimizer.cc:934] Skipping loop optimization for Merge node with control input: StatefulPartitionedCall/pretraining_model_1/scheduled_masking_layer_1/cond/branch_executed/_90



 --- End of Epoch 1: Teacher Forcing Ratio set to 0.96 ---
500/500 ━━━━━━━━━━━━━━━━━━━━ 29s 47ms/step - categorical_accuracy: 0.4028 - loss: 1.8496 - val_categorical_accuracy: 0.4607 - val_loss: 1.7168
Epoch 2/50
499/500 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - categorical_accuracy: 0.5071 - loss: 1.6327
 --- End of Epoch 2: Teacher Forcing Ratio set to 0.92 ---
500/500 ━━━━━━━━━━━━━━━━━━━━ 23s 46ms/step - categorical_accuracy: 0.5468 - loss: 1.5526 - val_categorical_accuracy: 0.5318 - val_loss: 1.5829
Epoch 3/50
499/500 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - categorical_accuracy: 0.6600 - loss: 1.3281
 --- End of Epoch 3: Teacher Forcing Ratio set to 0.88 ---
500/500 ━━━━━━━━━━━━━━━━━━━━ 23s 46ms/step - categorical_accuracy: 0.6946 - loss: 1.2678 - val_categorical_accuracy: 0.6464 - val_loss: 1.3641
Epoch 4/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - categorical_accuracy: 0.7848 - loss: 1.1086
 --- End of Epoch 4: Teacher Forcing Ratio set to 0.84 ---
500/500 ━━━━━━━━━━━━━━━━━━━━ 23s 46ms

In [16]:
# Fine tune training
## We choose a lower learning rate and higher weight decay.
learning_rate=1.0e-5
weight_decay=5.0e-2

history_fine_tune = train_fine_tune(image2text_pretraining,
                                    x=[X_train_pt, y_train_in_pt], 
                                    y=y_train_target_pt,
                                    val_data = ([X_val_pt, y_val_in_pt], y_val_target_pt),
                                    learning_rate=1.0e-5,
                                    weight_decay=5.0e-2
                                    )


Epoch 1/60


E0000 00:00:1767733801.039102 2154167 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: MutableGraphView::SortTopologically error: detected edge(s) creating cycle(s) {'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_100/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/Sigmoid_2' -> 'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_100/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/mul_6', 'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_100/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/Sigmoid' -> 'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_100/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/mul_5', 'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_100/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_ce

500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - categorical_accuracy: 0.9811 - loss: 0.6183

W0000 00:00:1767733823.540297 2154167 loop_optimizer.cc:934] Skipping loop optimization for Merge node with control input: StatefulPartitionedCall/pretraining_model_1/scheduled_masking_layer_1/cond/branch_executed/_90



 --- End of Epoch 1: Teacher Forcing Ratio set to 0.48 ---
500/500 ━━━━━━━━━━━━━━━━━━━━ 26s 45ms/step - categorical_accuracy: 0.9815 - loss: 0.6168 - val_categorical_accuracy: 0.9619 - val_loss: 0.6759 - learning_rate: 1.0000e-05
Epoch 2/60
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - categorical_accuracy: 0.9819 - loss: 0.6156
 --- End of Epoch 2: Teacher Forcing Ratio set to 0.46 ---
500/500 ━━━━━━━━━━━━━━━━━━━━ 22s 45ms/step - categorical_accuracy: 0.9824 - loss: 0.6145 - val_categorical_accuracy: 0.9613 - val_loss: 0.6777 - learning_rate: 1.0000e-05
Epoch 3/60
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - categorical_accuracy: 0.9826 - loss: 0.6142
 --- End of Epoch 3: Teacher Forcing Ratio set to 0.44 ---
500/500 ━━━━━━━━━━━━━━━━━━━━ 22s 44ms/step - categorical_accuracy: 0.9825 - loss: 0.6138 - val_categorical_accuracy: 0.9608 - val_loss: 0.6779 - learning_rate: 1.0000e-05
Epoch 4/60
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - categorical_accuracy: 0.9826 - loss: 0.6130
 --- End of 

In [30]:
# Export so that it can be used in A2_RNNs_Joep_full_model.ipynb without ruining the current optimal weights
image2text_pretraining.summary()
image2text_pretraining.save('visual_encoder.keras')

Model: "pretraining_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence            │ (None, 5, 28, 28, │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_model       │ [(None, 5, 7, 7,  │    906,560 │ sequence[0][0]    │
│ (Functional)        │ 128), (None, 7,   │            │                   │
│                     │ 7, 128), (None,   │            │                   │
│                     │ 7, 7, 128)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer         │ (None, 6, 15)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ h_flattened         │ (None, 128)       │          0 │ encoder_model[0]… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ c_flattened         │ (None, 128)       │          0 │ encoder_model[0]… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ scheduled_masking_… │ (None, 4, 15)     │          0 │ input_layer[0][0] │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ h0 (Dense)          │ (None, 256)       │     33,024 │ h_flattened[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ c0 (Dense)          │ (None, 256)       │     33,024 │ c_flattened[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_pre_training   │ [(None, 4, 256),  │    278,528 │ scheduled_maskin… │
│ (LSTM)              │ (None, 256),      │            │ h0[0][0],         │
│                     │ (None, 256)]      │            │ c0[0][0]          │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_13 │ (None, 4, 15)     │      3,855 │ lstm_pre_trainin… │
│ (TimeDistributed)   │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,764,591 (14.36 MB)

 Trainable params: 1,254,799 (4.79 MB)

 Non-trainable params: 192 (768.00 B)

 Optimizer params: 2,509,600 (9.57 MB)

### Inference

In [17]:
def predict_expressions_autoregressive(model, X_images, seq_len=6, vocab_size=15):
    """
    Performs greedy autoregressive decoding for the visual model.
    """
    batch_size = tf.shape(X_images)[0]
    
    # Initialize an empty sequence (zeros) as the starting point for Y_in
    # Shape: [Batch, Seq_Len, Vocab_Size]
    generated_seq = np.zeros((batch_size, seq_len, vocab_size), dtype='float32')
    
    # Generate tokens one by one
    for t in range(seq_len):
        # Predict the distribution for all timesteps based on current knowledge
        preds = model.predict([X_images, generated_seq], verbose=0)
        
        # We only care about the prediction at the current timestep 't'
        current_probs = preds[:, t, :]
        predicted_indices = np.argmax(current_probs, axis=-1)
        
        # Fill the NEXT timestep in generated_seq with a one-hot vector
        # (This simulates the Teacher Forcing input for the next step)
        if t < seq_len - 1:
            for i in range(batch_size):
                generated_seq[i, t + 1, predicted_indices[i]] = 1.0
                
    # Return the final indices for comparison
    return np.argmax(preds, axis=-1)

def evaluate_visual_encoder(model, X_test, batch_size=64):
    """
    Batched inference loop for the entire test set.
    """
    all_predictions = []
    num_samples = X_test.shape[0]
    
    for i in range(0, num_samples, batch_size):
        X_batch = X_test[i : i + batch_size]
        batch_preds = predict_expressions_autoregressive(model, X_batch)
        all_predictions.append(batch_preds)
        
        if (i // batch_size) % 5 == 0:
            print(f"Inference Progress: {i}/{num_samples}")
            
    return np.concatenate(all_predictions, axis=0)

In [19]:
import numpy as np

def calculate_visual_metrics(y_true_indices, y_pred_indices):
    """
    Calculates Token-level and Sequence-level accuracy.
    
    Args:
        y_true_indices: Array of shape [Samples, Seq_Len] containing ground truth.
        y_pred_indices: Array of shape [Samples, Seq_Len] containing predictions.
    """
    y_true = np.array(y_true_indices)
    y_pred = np.array(y_pred_indices)

    # 1. Token Accuracy (Pointwise)
    # Measures the percentage of individual digits/operators correctly identified.
    correct_tokens = (y_true == y_pred)
    token_acc = np.mean(correct_tokens) * 100

    # 2. Expression Accuracy (Exact Match / Sequence Accuracy)
    # Measures if the entire mathematical expression is correct.
    # In math, a single wrong token makes the entire calculation fail.
    exact_match = np.all(correct_tokens, axis=1)
    sequence_acc = np.mean(exact_match) * 100

    # 3. Positional Accuracy (Drift Analysis)
    # Checks if the model gets worse as the sequence progresses (Exposure Bias).
    positional_acc = np.mean(correct_tokens, axis=0) * 100

    print(f"{'='*30}")
    print(f"VISUAL ENCODER PERFORMANCE")
    print(f"{'='*30}")
    print(f"Overall Token Accuracy:    {token_acc:.2f}%")
    print(f"Exact Expression Match:     {sequence_acc:.2f}%")
    print(f"{'-'*30}")
    print(f"Positional Accuracy (Time-step):")
    for i, acc in enumerate(positional_acc):
        print(f"  Step {i+1}: {acc:.2f}%")
    print(f"{'='*30}")

    return {
        "token_acc": token_acc,
        "sequence_acc": sequence_acc,
        "positional_acc": positional_acc
    }

In [20]:
# --- 1. Generate Predictions ---
print(f"Running inference on {len(X_test_pt)} samples...")
y_pred_indices = evaluate_visual_encoder(image2text_pretraining, X_test_pt)

# --- 2. Align Ground Truth ---
# Your target is in y_test_target_pt (One-Hot)
y_true_indices = np.argmax(y_test_target_pt, axis=-1)

# FORCE ALIGNMENT: 
# This slices the longer array to match the shorter one (2000 vs 2001)
min_N = min(len(y_true_indices), len(y_pred_indices))
y_true_final = y_true_indices[:min_N]
y_pred_final = y_pred_indices[:min_N]

print(f"Successfully aligned to {min_N} samples.")

# --- 3. Calculate and Display Metrics ---
# Uses the function calculate_visual_metrics defined in your code
results = calculate_visual_metrics(y_true_final, y_pred_final)

Running inference on 2000 samples...
Inference Progress: 0/2000
Inference Progress: 320/2000
Inference Progress: 640/2000
Inference Progress: 960/2000
Inference Progress: 1280/2000
Inference Progress: 1600/2000
Inference Progress: 1920/2000
Successfully aligned to 2000 samples.
VISUAL ENCODER PERFORMANCE
Overall Token Accuracy:    96.69%
Exact Expression Match:     81.45%
------------------------------
Positional Accuracy (Time-step):
  Step 1: 96.25%
  Step 2: 95.50%
  Step 3: 99.40%
  Step 4: 95.00%
  Step 5: 94.00%
  Step 6: 100.00%


# Calculator model

## Building calculator

In [21]:
# Calculator model
def build_text2text_calc(dropout = 0.5, max_size=512, max_answer_length_tf=4,RLstrength=1.0e-4):

    vocab_size = len(vocabulary_tf)

    # Define input layer of full model
    X_in = Input(shape = (6, vocab_size), name = 'expression_input')
    Y_in = Input(shape=(max_answer_length_tf, len(vocabulary_tf)), name = "answer")
    masked_ground_truth = Lambda(
    lambda x, training: scheduled_mask_layer(x, training=training, prob=sampling_prob),
    output_shape=(4, 15), # Note: Use (4, 15) for calculator, (6, 15) for visual encoder
    name="scheduled_masking_layer"
    )(Y_in)

    # calculator encoder
    encoder_lstm = LSTM(max_size, return_state=True, return_sequences=True, name = 'calculator_encoder')
    key, hidden, cell = encoder_lstm(X_in)
    ini_state = [hidden, cell]    

    ## Calculator LSTM
    decoder_lstm = LSTM(
        max_size, 
        return_sequences = True, 
        return_state=True, 
        dropout = dropout, 
        recurrent_dropout = dropout, 
        name='decoder_lstm',
        kernel_regularizer = L2(RLstrength/4),
        recurrent_regularizer = L2(RLstrength/4)
        )
    
    # Calculator decoder
    query, _, _ = decoder_lstm(masked_ground_truth, initial_state = ini_state)

    attention_block = Attention(name='attention_block')([query, key])

    combined = Concatenate(axis=-1, name = 'concat_q_A')([query, attention_block])
    combined = Dense(max_size, name = 'combined_dense', kernel_regularizer = L1L2(l1 = 5.0e-5, l2=RLstrength/2))(combined)

    residual = TimeDistributed(Dense(
        max_size, 
        use_bias=False, 
        kernel_regularizer = L2(RLstrength/2)
        ), 
        name = 'decoder_res_dense'
        )(Y_in)

    final = Add(name= 'decoder_with_residual')([combined, residual])
    final = TimeDistributed(Activation('relu'), name = "decoder_activation")(final)
    final = LayerNormalization(axis=-1, name='decoder_layer_norm')(final)
    
    y_out = TimeDistributed(Dense(vocab_size, activation='softmax'), name='decoder_dense')(final)


    # Full model step
    full = tf.keras.Model(inputs=[X_in, Y_in], outputs = y_out, name = 'calculator')
#    full.compile(
#        loss='categorical_crossentropy', optimizer=Adam(learning_rate=learning_rate), metrics=['categorical_accuracy']
#    )

    full.summary(expand_nested=True)
    return full

## Training calculator

In [23]:
# Training the calculator

## Building model
dropout=0.5
RLstrength=0
text2text_calculator = build_text2text_calc(dropout=dropout, RLstrength=RLstrength, max_size=max_size)
sampling_prob.assign(1.0)

learning_rate=5.0e-4 
weight_decay=1.0e-4
history_warmup = train_warmup(text2text_calculator, 
                              x=[X_train_calc, y_train_in_calc], 
                              y = y_train_target_calc, 
                              val_data = ([X_val_calc, y_val_in_calc], y_val_target_calc),
                              learning_rate=learning_rate,
                              weight_decay = weight_decay)

Model: "calculator"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ answer (InputLayer) │ (None, 4, 15)     │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expression_input    │ (None, 6, 15)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ scheduled_masking_… │ (None, 4, 15)     │          0 │ answer[0][0]      │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ calculator_encoder  │ [(None, 6, 256),  │    278,528 │ expression_input… │
│ (LSTM)              │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 4, 256),  │    278,528 │ scheduled_maskin… │
│                     │ (None, 256),      │            │ calculator_encod… │
│                     │ (None, 256)]      │            │ calculator_encod… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_block     │ (None, 4, 256)    │          0 │ decoder_lstm[0][… │
│ (Attention)         │                   │            │ calculator_encod… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_q_A          │ (None, 4, 512)    │          0 │ decoder_lstm[0][… │
│ (Concatenate)       │                   │            │ attention_block[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ combined_dense      │ (None, 4, 256)    │    131,328 │ concat_q_A[0][0]  │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_res_dense   │ (None, 4, 256)    │      3,840 │ answer[0][0]      │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_with_resid… │ (None, 4, 256)    │          0 │ combined_dense[0… │
│ (Add)               │                   │            │ decoder_res_dens… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_activation  │ (None, 4, 256)    │          0 │ decoder_with_res… │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_layer_norm  │ (None, 4, 256)    │        512 │ decoder_activati… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_dense       │ (None, 4, 15)     │      3,855 │ decoder_layer_no… │
│ (TimeDistributed)   │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 696,591 (2.66 MB)

 Trainable params: 696,591 (2.66 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50


W0000 00:00:1767735030.205916 2154167 loop_optimizer.cc:934] Skipping loop optimization for Merge node with control input: StatefulPartitionedCall/calculator_1/scheduled_masking_layer_1/cond/branch_executed/_61


497/500 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - categorical_accuracy: 0.4956 - loss: 2.1330

W0000 00:00:1767735038.354723 2154167 loop_optimizer.cc:934] Skipping loop optimization for Merge node with control input: StatefulPartitionedCall/calculator_1/scheduled_masking_layer_1/cond/branch_executed/_33



 --- End of Epoch 1: Teacher Forcing Ratio set to 0.96 ---
500/500 ━━━━━━━━━━━━━━━━━━━━ 10s 17ms/step - categorical_accuracy: 0.5389 - loss: 1.9448 - val_categorical_accuracy: 0.5662 - val_loss: 1.7767
Epoch 2/50
498/500 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - categorical_accuracy: 0.5774 - loss: 1.7272
 --- End of Epoch 2: Teacher Forcing Ratio set to 0.92 ---
500/500 ━━━━━━━━━━━━━━━━━━━━ 8s 16ms/step - categorical_accuracy: 0.5862 - loss: 1.6831 - val_categorical_accuracy: 0.6166 - val_loss: 1.5836
Epoch 3/50
499/500 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - categorical_accuracy: 0.6224 - loss: 1.5562
 --- End of Epoch 3: Teacher Forcing Ratio set to 0.88 ---
500/500 ━━━━━━━━━━━━━━━━━━━━ 8s 17ms/step - categorical_accuracy: 0.6309 - loss: 1.5230 - val_categorical_accuracy: 0.6431 - val_loss: 1.4640
Epoch 4/50
498/500 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - categorical_accuracy: 0.6549 - loss: 1.4404
 --- End of Epoch 4: Teacher Forcing Ratio set to 0.84 ---
500/500 ━━━━━━━━━━━━━━━━━━━━ 8s 16ms/st

In [27]:
learning_rate=2.5000e-04#5.0e-4 
weight_decay=1.0e-5 

history_fine_tune = train_fine_tune(text2text_calculator,
                x=[X_train_calc, y_train_in_calc], 
                y=y_train_target_calc,
                val_data=([X_val_calc, y_val_in_calc], y_val_target_calc),
                learning_rate=learning_rate,
                weight_decay=weight_decay)

Epoch 1/60


W0000 00:00:1767736169.177368 2154167 loop_optimizer.cc:934] Skipping loop optimization for Merge node with control input: StatefulPartitionedCall/calculator_1/scheduled_masking_layer_1/cond/branch_executed/_61


498/500 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - categorical_accuracy: 0.9993 - loss: 0.5800

W0000 00:00:1767736177.523710 2154167 loop_optimizer.cc:934] Skipping loop optimization for Merge node with control input: StatefulPartitionedCall/calculator_1/scheduled_masking_layer_1/cond/branch_executed/_33



 --- End of Epoch 1: Teacher Forcing Ratio set to 0.05 ---
500/500 ━━━━━━━━━━━━━━━━━━━━ 10s 17ms/step - categorical_accuracy: 0.9996 - loss: 0.5794 - val_categorical_accuracy: 0.9996 - val_loss: 0.5757 - learning_rate: 2.5000e-04
Epoch 2/60
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - categorical_accuracy: 0.9995 - loss: 0.5792
 --- End of Epoch 2: Teacher Forcing Ratio set to 0.05 ---
500/500 ━━━━━━━━━━━━━━━━━━━━ 8s 17ms/step - categorical_accuracy: 0.9994 - loss: 0.5798 - val_categorical_accuracy: 0.9993 - val_loss: 0.5790 - learning_rate: 2.5000e-04
Epoch 3/60
499/500 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - categorical_accuracy: 0.9993 - loss: 0.5810
 --- End of Epoch 3: Teacher Forcing Ratio set to 0.05 ---
500/500 ━━━━━━━━━━━━━━━━━━━━ 8s 16ms/step - categorical_accuracy: 0.9993 - loss: 0.5805 - val_categorical_accuracy: 0.9995 - val_loss: 0.5762 - learning_rate: 2.5000e-04
Epoch 4/60
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - categorical_accuracy: 0.9998 - loss: 0.5784
 --- End of Ep

In [ ]:
# Old
"""
### Recompile so AdamW momenta are reset
learning_rate=5.0e-4 
weight_decay=1.0e-5 

optimizer = tf.keras.optimizers.AdamW(
    learning_rate=learning_rate,
    weight_decay=weight_decay,
)

text2text_calculator.compile(
    loss=loss, 
    optimizer=optimizer, 
    metrics=['categorical_accuracy'])

scheduler_patience = 3
stopper_patience = 12

early_stopper = tf.keras.callbacks.EarlyStopping(
    monitor = 'val_loss',
    patience=stopper_patience,
    restore_best_weights=True
)

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=scheduler_patience,
    min_lr=1e-6,
    verbose=1
)

### Fine-tune training
history_calc = text2text_calculator.fit(x=[X_train_calc, y_train_in_calc], y=y_train_target_calc, 
               epochs = 60,
               batch_size = 32,
               validation_data = ([X_val_calc, y_val_in_calc], y_val_target_calc),
               callbacks=[lr_scheduler, early_stopper],
               verbose=1)
"""

In [31]:
# Export so that it can be used in A2_RNNs_Joep_full_model.ipynb without ruining the current optimal weights
text2text_calculator.summary()
text2text_calculator.save('calculator.keras')

Model: "calculator"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ answer (InputLayer) │ (None, 4, 15)     │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expression_input    │ (None, 6, 15)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ scheduled_masking_… │ (None, 4, 15)     │          0 │ answer[0][0]      │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ calculator_encoder  │ [(None, 6, 256),  │    278,528 │ expression_input… │
│ (LSTM)              │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 4, 256),  │    278,528 │ scheduled_maskin… │
│                     │ (None, 256),      │            │ calculator_encod… │
│                     │ (None, 256)]      │            │ calculator_encod… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_block     │ (None, 4, 256)    │          0 │ decoder_lstm[0][… │
│ (Attention)         │                   │            │ calculator_encod… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_q_A          │ (None, 4, 512)    │          0 │ decoder_lstm[0][… │
│ (Concatenate)       │                   │            │ attention_block[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ combined_dense      │ (None, 4, 256)    │    131,328 │ concat_q_A[0][0]  │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_res_dense   │ (None, 4, 256)    │      3,840 │ answer[0][0]      │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_with_resid… │ (None, 4, 256)    │          0 │ combined_dense[0… │
│ (Add)               │                   │            │ decoder_res_dens… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_activation  │ (None, 4, 256)    │          0 │ decoder_with_res… │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_layer_norm  │ (None, 4, 256)    │        512 │ decoder_activati… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_dense       │ (None, 4, 15)     │      3,855 │ decoder_layer_no… │
│ (TimeDistributed)   │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,089,775 (7.97 MB)

 Trainable params: 696,591 (2.66 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,393,184 (5.31 MB)

## Inference

In [28]:
def predict_calculator_autoregressive(model, X_expressions, seq_len=4, vocab_size=15):
    """
    Performs greedy autoregressive decoding for the calculator model.
    
    Args:
        model: The trained text2text_calculator model.
        X_expressions: The input math expressions (One-Hot, shape: [Batch, 6, 15]).
        seq_len: Expected length of the answer (default 4).
        vocab_size: Size of the vocabulary (default 15).
    """
    batch_size = tf.shape(X_expressions)[0]
    
    # Initialize an empty sequence (zeros) for the 'answer' input layer
    # Shape: [Batch, seq_len, vocab_size]
    generated_ans = np.zeros((batch_size, seq_len, vocab_size), dtype='float32')
    
    # Iterate through each time step of the answer
    for t in range(seq_len):
        # Predict the next token based on the input expression and generated answer so far
        preds = model.predict([X_expressions, generated_ans], verbose=0)
        
        # Take the prediction for the CURRENT timestep t
        current_step_probs = preds[:, t, :]
        predicted_indices = np.argmax(current_step_probs, axis=-1)
        
        # Update the sequence for the NEXT timestep
        # We place a 1.0 at the predicted index to create a one-hot vector for the next step
        if t < seq_len - 1:
            for i in range(batch_size):
                generated_ans[i, t + 1, predicted_indices[i]] = 1.0
                
    # Return the final sequence of indices
    return np.argmax(preds, axis=-1)

def evaluate_calculator(model, X_test, batch_size=64):
    """
    Batched inference loop for the calculator test set.
    """
    all_predictions = []
    num_samples = X_test.shape[0]
    
    for i in range(0, num_samples, batch_size):
        X_batch = X_test[i : i + batch_size]
        batch_preds = predict_calculator_autoregressive(model, X_batch)
        all_predictions.append(batch_preds)
        
        if (i // batch_size) % 10 == 0:
            print(f"Calculator Inference: {i}/{num_samples}")
            
    return np.concatenate(all_predictions, axis=0)

In [29]:
# --- 1. Generate Predictions ---
# Note: X_test_calc contains the one-hot expressions (e.g., '10+10')
print(f"Starting calculator evaluation on {len(X_test_calc)} samples...")
y_pred_calc_indices = evaluate_calculator(text2text_calculator, X_test_calc)

# --- 2. Convert Ground Truth to Indices ---
# Your targets are in y_test_target_calc (One-Hot, shape [N, 4, 15])
y_true_calc_indices = np.argmax(y_test_target_calc, axis=-1)

# --- 3. Synchronize Lengths (Safety Slice) ---
# This ensures both arrays have the same sample count (N)
min_N = min(len(y_true_calc_indices), len(y_pred_calc_indices))
y_true_final = y_true_calc_indices[:min_N]
y_pred_final = y_pred_calc_indices[:min_N]

print(f"Aligned calculator data to {min_N} samples.")

# --- 4. Calculate and Display Metrics ---
# We can reuse the same metrics function used for the visual encoder
results_calc = calculate_visual_metrics(y_true_final, y_pred_final)

Starting calculator evaluation on 2000 samples...


W0000 00:00:1767736657.584147 2154167 loop_optimizer.cc:934] Skipping loop optimization for Merge node with control input: calculator_1/scheduled_masking_layer_1/cond/branch_executed/_8


Calculator Inference: 0/2000
Calculator Inference: 640/2000
Calculator Inference: 1280/2000
Calculator Inference: 1920/2000
Aligned calculator data to 2000 samples.
VISUAL ENCODER PERFORMANCE
Overall Token Accuracy:    87.91%
Exact Expression Match:     58.75%
------------------------------
Positional Accuracy (Time-step):
  Step 1: 58.75%
  Step 2: 93.45%
  Step 3: 99.45%
  Step 4: 100.00%


W0000 00:00:1767736662.389546 2154167 loop_optimizer.cc:934] Skipping loop optimization for Merge node with control input: calculator_1/scheduled_masking_layer_1/cond/branch_executed/_8
